# OTOCs of the memory: plots

Loads the OTOC results written by `OTOCS_poster_init_coherent.ipynb` from
`results/` and plots them. Nothing is simulated here, so this needs neither the
cluster nor jax.

- `otoc_example_<date>.json`, `otoc_example_stable_<date>.json`: `C(t) =
  -<[x_a(t), p_a(0)]^2>` at `epsilon_0 = 0.93` (chaotic) and `0.2` (stabilized),
  `epsilon_d = 10 g2`, `n_a = 38`.
- `lyap_array_<date>.json`, `R_sq_array_<date>.json`, `end_otoc_array_<date>.json`:
  the heatmap over `epsilon_0` and `epsilon_d / g2` at `n_a = 29`, `n_b = 17`. These
  hold the rate `m / 2` and the `R^2` of the linear fit of `log C` against `kappa_b
  t`, and `C(T)`.

The JSONs hold only the values. The heatmap files are a list of rows, one per
`epsilon_0`, one entry per `epsilon_d / g2`. The example curves have no time
axis, so it is rebuilt from the curve's length, which tells the two grids apart:

- **49 points:** the adjoint-equation version (2026-10-12 on), with buffer loss.
  It uses 8 points per drive period `2 pi / w_b` over 6 periods, so `kappa_b T = 0.225`.
- **50 points:** the earlier closed-system version (also `OTOCS_poster.ipynb`).
  It uses `linspace(0, 0.2 / kappa, 50)`.

The two give different physics, so curves of different dates are not
comparable. `OTOCS_poster.ipynb` writes the same file names, so a run of it on
the same day overwrites these files.

If a grid changes in `OTOCS_poster_init_coherent.ipynb`, change it here too. The
checks below catch a mismatch in size, but not in values.

**`g2` changed on 2026-10-12.** The simulating notebooks now use `g2 =
sin(epsilon_0) E_J phi_a^2 phi_b` (static pump), in place of `J_1(epsilon_0) E_J
phi_a^2 phi_b`. Result files written before that change were computed with
`J_1`: in them, `epsilon_d / g2` is `epsilon_d / g2_J1`, and the true ratio is
`J_1 / sin` (about 1/2) times that, so their cats are about half the size the
axis says. A re-run on `12_10_2026` overwrites the files of that date, so check
the file's modification time when in doubt.

In [ ]:
import json
import pathlib
import re
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import Perso_colormap
import matplotlib.pyplot as plt
import numpy as np

# The results/ folder next to this notebook (or under the workspace root, if that
# is the kernel's working directory).
here = pathlib.Path.cwd()
results_dir = (here if here.name == "Quantum_chaos" else here / "Quantum_chaos") / "results"

# None: the most recent date for which a group's files all exist. Or a date,
# e.g. "12_10_2026".
DATE = None

EXAMPLE_FILES = ("otoc_example", "otoc_example_stable")
HEATMAP_FILES = ("lyap_array", "R_sq_array", "end_otoc_array")


def _dates(prefix):
    """Dates (dd_mm_yyyy) of the files <prefix>_<date>.json in results_dir."""
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d{{2}}_\d{{2}}_\d{{4}})\.json$")
    return {m.group(1) for p in results_dir.iterdir() if (m := pattern.match(p.name))}


def _date_key(date):
    d, m, y = map(int, date.split("_"))
    return (y, m, d)


def load_group(prefixes, date=DATE):
    """{prefix: array} for the files <prefix>_<date>.json, and the date used."""
    if date is None:
        common = set.intersection(*(_dates(p) for p in prefixes))
        if not common:
            raise FileNotFoundError(f"no date with all of {prefixes} in {results_dir}; "
                                    "run OTOCS_poster_init_coherent.ipynb first")
        date = max(common, key=_date_key)
    return {p: np.array(json.load(open(results_dir / f"{p}_{date}.json")), dtype=float)
            for p in prefixes}, date


examples, examples_date = load_group(EXAMPLE_FILES)
heatmap, heatmap_date = load_group(HEATMAP_FILES)
for date, group in ((examples_date, examples), (heatmap_date, heatmap)):
    for name, arr in group.items():
        print(f"loaded {name}_{date}.json: shape {arr.shape}, "
              f"{np.isnan(arr).sum()} failed point(s), "
              f"values in [{np.nanmin(arr):.4g}, {np.nanmax(arr):.4g}]")

In [ ]:
# Constants and grids of OTOCS_poster_init_coherent.ipynb.
w_a = 25.338776456203686
w_b = 2 * w_a
kappa = np.pi / 10.4
tau = 2 * np.pi / w_b  # drive period


def time_axis(n_points):
    """The t grid of an example curve, from its length (see the top of this notebook)."""
    n_periods, n_sub = int(np.ceil(.2 / kappa / tau)), 8
    if n_points == n_periods * n_sub + 1:   # adjoint equation, with buffer loss
        return tau / n_sub * np.arange(n_points), "adjoint equation, buffer loss"
    if n_points == 50:                      # closed system, before 2026-10-12
        return np.linspace(0, .2 / kappa, n_points), "closed system (old)"
    raise ValueError(f"no known time grid with {n_points} points")


# The heatmap grid (the same in OTOCS_poster.ipynb).
epsilon_0_list = list(np.concatenate((np.arange(0.2, .6, 0.07), np.arange(0.6, 1.1, 0.03))))
epsilon_0_arr = np.array(epsilon_0_list, dtype=np.float32)
ratio_list = np.arange(2, 10, 1)
n_a, n_b = 29, 17

lyap_arr = heatmap["lyap_array"]
R_sq_arr = heatmap["R_sq_array"]
end_otoc_arr = heatmap["end_otoc_array"]
for name, arr in heatmap.items():
    assert arr.shape == (len(epsilon_0_arr), len(ratio_list)), (
        f"{name}_{heatmap_date}.json is {arr.shape}, the grid here is "
        f"{(len(epsilon_0_arr), len(ratio_list))}: the two notebooks disagree on the grid")

## Chaotic and stabilized regimes

`C(t)` on a log scale against `kappa_b t`, with the linear fit of `log C` whose
slope `m` is the growth rate, as in `OTOCS_poster_init_coherent.ipynb`.

In [ ]:
def fit_log_otoc(t, otoc):
    """Linear fit of log C(t) against kappa_b t: slope m, intercept, R^2."""
    t_kappa = t * kappa
    log_otoc = np.log(otoc)
    (m, c), residuals, _, _, _ = np.polyfit(t_kappa, log_otoc, 1, full=True)
    R_sq = 1 - residuals.item() / np.sum((log_otoc - np.mean(log_otoc))**2)
    return m, c, R_sq


fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True, sharey=True)
for ax, (name, title) in zip(axes, (("otoc_example", r"chaotic, $\varepsilon_p = 0.93$"),
                                    ("otoc_example_stable", r"stabilized, $\varepsilon_p = 0.2$"))):
    otoc = examples[name]
    t, grid = time_axis(len(otoc))
    m, c, R_sq = fit_log_otoc(t, otoc)
    ax.plot(t * kappa, otoc, "o-", ms=3, label=r"$C(t)$")
    ax.plot(t * kappa, np.exp(m * t * kappa + c), "--", color="0.4",
            label=rf"fit: $m = {m:.3g}$, $R^2 = {R_sq:.3g}$")
    ax.set_yscale("log")
    ax.set_xlabel(r"$\kappa_b t$")
    ax.set_title(f"{title}, " + r"$n_a=38$" + f"\n{grid}, {examples_date}", fontsize=9)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)
    print(f"{name:<20} m = {m:.6g}   R^2 = {R_sq:.6g}   ({grid})")
axes[0].set_ylabel(r"$C(t) = -\langle[x_a(t),p_a(0)]^2\rangle$")
plt.show()

## Heatmaps over the pump amplitude and the drive strength

`epsilon_0` is not evenly spaced (steps of `0.07` up to `0.6`, then `0.03`),
so the cells are drawn at their true positions with `pcolormesh`, as in
`plot_IPR.ipynb`. `C(T)` spans more than a decade, so it is on a log colour
scale.

In [ ]:
from matplotlib.colors import LogNorm


def _edges(x):
    """Cell edges around the centers x: the midpoints, extended at both ends."""
    x = np.asarray(x, dtype=float)
    mid = (x[1:] + x[:-1]) / 2
    return np.concatenate(([x[0] - (mid[0] - x[0])], mid, [x[-1] + (x[-1] - mid[-1])]))


fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.4), constrained_layout=True)
for ax, arr, label, kw in (
        (axes[0], lyap_arr, r"$m/2$", dict(cmap=Perso_colormap)),
        (axes[1], R_sq_arr, r"$R^2$", dict(cmap=Perso_colormap, vmin=0, vmax=1)),
        (axes[2], end_otoc_arr, r"$C(T)$", dict(cmap=Perso_colormap, norm=LogNorm()))):
    im = ax.pcolormesh(_edges(epsilon_0_arr), _edges(ratio_list), arr.T, **kw)
    ax.set_xlabel(r"$\varepsilon_p$" + " (rad)")
    ax.set_ylabel(r"$\varepsilon_d / g_2$")
    ax.set_title(label + r", $n_a=$" + str(n_a) + ", " + r"$n_b=$" + str(n_b)
                 + f", {heatmap_date}", fontsize=10)
    fig.colorbar(im, ax=ax, label=label)
plt.show()

## Cuts at fixed drive strength

The same data, one curve per `epsilon_d / g2`, coloured from light (`2`) to
dark (`9`). Points whose fit has `R^2 < R_SQ_MIN` are drawn hollow: there the
rate `m / 2` is a poor summary of `C(t)`.

In [ ]:
R_SQ_MIN = 0.8

colors = Perso_colormap(np.linspace(0.9, 0.1, len(ratio_list)))

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), constrained_layout=True)
for j, (ratio, c) in enumerate(zip(ratio_list, colors)):
    good = R_sq_arr[:, j] >= R_SQ_MIN
    for ax, arr in ((axes[0], lyap_arr), (axes[1], end_otoc_arr)):
        ax.plot(epsilon_0_arr, arr[:, j], "-", color=c, lw=2,
                label=rf"$\varepsilon_d/g_2 = {ratio:g}$")
        ax.plot(epsilon_0_arr[good], arr[good, j], "o", color=c, ms=5)
        ax.plot(epsilon_0_arr[~good], arr[~good, j], "o", mfc="white", mec=c, ms=5)
axes[0].set_ylabel(r"$m/2$")
axes[1].set_ylabel(r"$C(T)$")
axes[1].set_yscale("log")
for ax in axes:
    ax.set_xlabel(r"$\varepsilon_p$" + " (rad)")
    ax.set_title(r"$n_a=$" + str(n_a) + ", " + r"$n_b=$" + str(n_b)
                 + f", {heatmap_date} (hollow: $R^2 < {R_SQ_MIN}$)", fontsize=10)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8, ncol=2)
plt.show()

## The rate `m / 2` against the two-photon coupling `g2`

The same `m / 2`, with the pump amplitude replaced by `g2 = sin(epsilon_0) E_J
phi_a^2 phi_b`. This is the `g2` that sets `epsilon_d = ratio g2` in
`OTOCS_poster_init_coherent.ipynb`, computed from the same float32 `epsilon_0`,
and shown as `g2 / 2pi` in MHz. `sin` increases over this range, so `g2` keeps
the order of `epsilon_0`. Hollow points have `R^2 < R_SQ_MIN`, as above.

In [ ]:
# Circuit constants of OTOCS_poster_init_coherent.ipynb.
E_J = 37.12 * 2 * np.pi * 1.5
phi_a, phi_b = 0.11, 0.204

g2_arr = np.sin(epsilon_0_arr) * E_J * phi_a**2 * phi_b   # rad/ns, as in the heatmap run
g2_list = g2_arr / (2 * np.pi) * 1e3                      # g2 / 2pi, MHz
assert np.all(np.diff(g2_list) > 0)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), constrained_layout=True)

im = axes[0].pcolormesh(_edges(g2_list), _edges(ratio_list), lyap_arr.T, cmap=Perso_colormap)
axes[0].set_ylabel(r"$\varepsilon_d / g_2$")
fig.colorbar(im, ax=axes[0], label=r"$m/2$")

for j, (ratio, c) in enumerate(zip(ratio_list, colors)):
    good = R_sq_arr[:, j] >= R_SQ_MIN
    axes[1].plot(g2_list, lyap_arr[:, j], "-", color=c, lw=2,
                 label=rf"$\varepsilon_d/g_2 = {ratio:g}$")
    axes[1].plot(g2_list[good], lyap_arr[good, j], "o", color=c, ms=5)
    axes[1].plot(g2_list[~good], lyap_arr[~good, j], "o", mfc="white", mec=c, ms=5)
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].set_ylabel(r"$m/2$")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8, ncol=2)

for ax in axes:
    ax.set_xlabel(r"$g_2 / 2\pi$ (MHz)")
    ax.set_title(r"$m/2$, $n_a=$" + str(n_a) + ", " + r"$n_b=$" + str(n_b)
                 + f", {heatmap_date}", fontsize=10)
plt.show()